# Занятие 24. Практика: признаки для цены квартиры (~90 мин)

Вы **пишете весь код сами**. Ячейку **«Дано»** не меняйте.

Задача — регрессия: предсказать **цену** квартиры (млн руб.) по объявлению. Модель для проверки идей — kNN-регрессия (занятие 21), метрика — **MAE**. Теория — `Урок_23_Feature_Engineering_Теория.ipynb` (занятие 23).

Ориентир по времени указан у каждого задания. Если застряли — идите дальше и вернитесь позже.

### Оценивание (30 баллов)

| № | Тема | Баллы |
|---|---|---:|
| 0 | Постановка | 2 |
| 1 | Split | 3 |
| 2 | Числовые признаки | 3 |
| 3 | Признаки из даты | 3 |
| 4 | Категории | 4 |
| 5 | Пропуски | 3 |
| 6 | Baseline-набор и kNN | 4 |
| 7 | Добавляем группы признаков | 4 |
| 8 | Комбинированный набор | 2 |
| 9 | Итог | 2 |

---
## Дано: датасет

Синтетическая таблица объявлений (300 строк): площадь, комнаты, район, состояние, балкон, дата публикации, просмотры за месяц, доход района (есть пропуски) и **цена** — целевая переменная. Момент прогноза — `PREDICTION_DATE`.

In [ ]:
import numpy as np
import pandas as pd

rng = np.random.default_rng(42)
n = 300

districts = rng.choice(['центр', 'спальный', 'пригород'], size=n, p=[0.3, 0.5, 0.2])
condition = rng.choice(['требует ремонта', 'среднее', 'хорошее'], size=n, p=[0.2, 0.5, 0.3])
area = rng.uniform(25, 130, n).round(0)
rooms = np.clip((area / 30).astype(int) + rng.integers(0, 2, n), 1, 5)
balcony = rng.integers(0, 2, n)
listing_date = pd.Timestamp('2026-01-01') + pd.to_timedelta(rng.integers(0, 180, n), unit='D')
views = np.expm1(rng.normal(4.0, 1.2, n)).round(0)

district_bonus = pd.Series(districts).map({'центр': 4.0, 'спальный': 1.0, 'пригород': 0.0}).values
condition_bonus = pd.Series(condition).map(
    {'требует ремонта': -1.5, 'среднее': 0.0, 'хорошее': 1.5}
).values
price = (0.13 * area + 0.02 * area * balcony + district_bonus
         + condition_bonus + rng.normal(0, 1.0, n)).round(1)

district_income = pd.Series(districts).map({'центр': 95.0, 'спальный': 60.0, 'пригород': 45.0}).values
district_income = district_income + rng.normal(0, 5, n).round(0)
missing_mask = rng.random(n) < 0.15
district_income[missing_mask] = np.nan

flats = pd.DataFrame({
    'площадь': area,
    'комнаты': rooms,
    'район': districts,
    'состояние': condition,
    'балкон': balcony,
    'дата': listing_date,
    'просмотры_за_месяц': views,
    'доход_района': district_income,
    'цена': price,
})

PREDICTION_DATE = pd.Timestamp('2026-07-01')
RANDOM_STATE = 42
print('Объектов:', len(flats))
flats.head()


---
## Задание 0. Постановка (~5 мин) — **2 балла**

Перед тем как строить признаки, зафиксируйте постановку задачи обычным текстом. Напишите в markdown-ячейке, что одной строкой данных является объявление о квартире, какую величину мы хотим предсказывать и почему такая цель делает задачу регрессией. Отдельно укажите, в какой момент времени модель должна делать прогноз: это важно, потому что признаки должны быть доступны именно к этому моменту.

Завершите ответ примером утечки данных. Можно разобрать признак `цена_за_метр = цена / площадь`: если `цена` является целевой переменной, такой признак уже содержит часть правильного ответа и поэтому не может использоваться при обучении модели.

---
## Задание 1. Split (~7 мин) — **3 балла**

Сделайте первое рабочее разбиение данных: отделите от таблицы `flats` обучающую часть `flats_train` и валидационную часть `flats_val`. Используйте соотношение 70/30 и `random_state=RANDOM_STATE`, чтобы результат был воспроизводимым.

После разбиения выведите размеры обеих таблиц и проверьте, что дальше все обучаемые преобразования будут настраиваться только по `flats_train`. Валидационная часть нужна для честной проверки качества, а не для подглядывания при подготовке признаков.

---
## Задание 2. Числовые признаки (~10 мин) — **3 балла**

Добавьте в обе таблицы, `flats_train` и `flats_val`, два числовых признака. Первый признак, `площадь_на_комнату`, должен показывать, сколько квадратных метров в среднем приходится на одну комнату. Второй признак, `log_просмотры`, должен быть логарифмом `просмотры_за_месяц` через `np.log1p`, чтобы большие значения просмотров не слишком сильно доминировали.

После создания признаков убедитесь, что новые столбцы появились в обеих таблицах и в них нет пропусков. Эти признаки не требуют обучения на данных, поэтому одну и ту же формулу можно применить к train и validation.

---
## Задание 3. Признаки из даты (~10 мин) — **3 балла**

Преобразуйте дату публикации объявления в простые числовые признаки, которые модель сможет использовать без работы с типом datetime. В обеих таблицах выделите номер месяца в столбец `месяц`: это позволит модели учитывать сезонность объявлений хотя бы в базовом виде.

Также посчитайте `дней_с_публикации`: это разница между `PREDICTION_DATE` и датой публикации объявления. Проверьте, что получившееся число неотрицательное для всех строк, иначе это означало бы ошибку во времени прогноза или в данных.

---
## Задание 4. Категории (~12 мин) — **4 балла**

Закодируйте категориальные признаки так, чтобы их можно было передать в модель. Для признака `район` используйте `OneHotEncoder(handle_unknown='ignore')`: обучите кодировщик только на `flats_train`, а затем примените его и к train, и к validation. Такой режим нужен, чтобы новая или редкая категория в validation не ломала код.

Для признака `состояние` используйте `OrdinalEncoder` с явным порядком категорий: `требует ремонта < среднее < хорошее`. Добавьте в обе таблицы столбец `состояние_код` и проверьте, что порядок отражает смысл признака, а не случайный алфавитный порядок.

---
## Задание 5. Пропуски (~8 мин) — **3 балла**

Обработайте пропуски в признаке `доход_района` так, чтобы модель получила и заполненное значение, и информацию о самом факте пропуска. В обеих таблицах создайте индикатор `доход_пропущен`, где 1 означает, что исходное значение было пропущено, а 0 — что оно было известно.

Затем создайте `доход_заполнен`: замените пропуски медианой, посчитанной только по `flats_train`. Не пересчитывайте медиану на validation, потому что в рабочем сценарии параметры обработки должны приходить из train и затем одинаково применяться к новым данным.

---
## Задание 6. Baseline-набор и kNN (~10 мин) — **4 балла**

Постройте первый честный baseline для регрессии. В качестве минимального набора признаков возьмите только `площадь` и `комнаты`: это простая точка отсчёта, с которой потом можно сравнивать новые признаки.

Соберите `Pipeline`, где сначала стоит `StandardScaler`, а затем `KNeighborsRegressor(n_neighbors=5)`. Обучите pipeline на train и посчитайте MAE на validation, сохранив результат в `mae_base`. Pipeline здесь важен: масштабирование должно обучаться на train и затем теми же параметрами применяться к validation.

---
## Задание 7. Добавляем группы признаков (~12 мин) — **4 балла**

Проверьте, какие группы признаков действительно помогают baseline-модели. Для каждой группы соберите набор «базовые признаки плюс одна новая группа», обучите ту же kNN-модель и измерьте MAE на validation.

Сравните три варианта: числовые признаки `площадь_на_комнату` и `log_просмотры`; признаки `состояние_код` и `балкон`; one-hot столбцы районов. Результат оформите таблицей «набор признаков → MAE», чтобы по числам было видно, какие идеи улучшили модель, а какие нет.

---
## Задание 8. Комбинированный набор (~8 мин) — **2 балла**

Соберите финальный набор признаков не автоматически «всё подряд», а по результатам предыдущего сравнения. Начните с базовых признаков и добавьте только те группы, которые улучшили MAE в задании 7.

После этого снова обучите модель на train, посчитайте `mae_final` на validation и сравните его с `mae_base`. Важно показать не только число, но и логику выбора: какие группы вошли в финальный набор и почему.

---
## Задание 9. Итог (~8 мин) — **2 балла**

Напишите итоговый вывод в markdown-ячейке. Сначала перечислите группы признаков, которые улучшили MAE, и группы, которые не дали улучшения; объясните это как гипотезу, опираясь на таблицу из заданий 7–8.

Затем отдельно назовите признак, который вы не стали строить из-за утечки данных, и объясните, где именно он подглядывает ответ. В конце перечислите параметры обработки, которые нужно сохранить для применения модели к завтрашним объявлениям: обученные кодировщики категорий, медиану train для пропусков и параметры масштабирования внутри pipeline.